# 5. Validation: manual labelling of capability detections

**Methodology steps 3 and 4: qualitative analysis and validation.**

Regular expressions detect a *mention*, not an *instruction to execute*. We draw a stratified random sample (seed 42) of matched snippets per capability and label each by hand:

* `instructs` = the skill tells the agent to perform the action;
* `mentions` = the text refers to it without instructing (for example "never commit the API key");
* `false_match` = the pattern matched something unrelated.

Precision = `instructs` / labelled. The CSV goes to `results/tables/validation_sample.csv`; fill the `label` column by hand and re-run the last cell.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
print("database:", os.environ["GITSKILLS_DB"])
import re

In [ ]:
PER_CATEGORY = 12   # 9 categories x 12 = 108 snippets
skills = clean.strict(clean.load_skills())
rows = []
for cap, rx in patterns.COMPILED_RISKS.items():
    hits = skills[skills[f"risk_{cap}"].fillna(False)]
    pick = hits.sample(min(PER_CATEGORY, len(hits)), random_state=42)
    texts = db.query(f"SELECT file_sha, content FROM artifacts WHERE dedup_primary = 1 AND file_sha IN ({','.join('?' * len(pick))})",
                     tuple(pick["file_sha"]))
    for _, t in texts.iterrows():
        m = rx.search(t["content"] or "")
        if m:
            s = max(0, m.start() - 160); e = min(len(t["content"]), m.end() + 160)
            rows.append({"capability": cap, "file_sha": t["file_sha"], "snippet": t["content"][s:e].replace("\n", " "), "label": ""})
sample = pd.DataFrame(rows)
out = ROOT / "results/tables/validation_sample.csv"
if not out.exists():
    sample.to_csv(out, index=False)
print(f"{len(sample)} snippets -> {out}")
sample.head()

## Precision per capability (after labelling)

In [ ]:
lab = pd.read_csv(ROOT / "results/tables/validation_sample.csv").fillna({"label": ""})
done = lab[lab["label"] != ""]
if done.empty:
    print("No labels yet: fill the 'label' column in validation_sample.csv")
else:
    prec = done.groupby("capability")["label"].apply(lambda s: (s == "instructs").mean()).rename("precision")
    save_table(prec.reset_index(), "validation_precision")
    display(prec.to_frame())